Calculate the impervious share per grid

In [4]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from rasterstats import zonal_stats
from pathlib import Path
from tqdm import tqdm

# === 路径 ===
base_dir = Path("/Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000")
grid_path = base_dir / "Jiangsu_grid_500m_ID.gpkg"

out_csv  = base_dir / "Jiangsu_grid_impervious_2012_2022.csv"
out_gpkg = base_dir / "Jiangsu_grid_impervious_2012_2022.gpkg"

# === 参数 ===
years = range(2012, 2023)
cell_area = 500 * 500  # 每格面积 (m²)

# === Step 1. 读取 grid ===
grid = gpd.read_file(grid_path)
print(f"✅ Loaded grid: {len(grid)} cells")

# === Step 2. 逐年计算 impervious (value = 8) ===
all_years = []

for year in tqdm(years, desc="Processing CLCD yearly impervious area"):
    clcd_path = base_dir / f"CLCD_v01_{year}_albert_jiangsu_albers.tif"
    print(f"  → {year}: {clcd_path.name}")
    
    # zonal_stats: 统计每个 polygon 内的各值数量
    stats = zonal_stats(
        vectors=grid,
        raster=str(clcd_path),
        categorical=True,
        nodata=0
    )
    
    df = pd.DataFrame(stats).fillna(0)
    
    # impervious = value 8
    if 8 in df.columns:
        imperv_frac = df[8] / df.sum(axis=1)
        imperv_area = df[8] * (cell_area / df.sum(axis=1))
    else:
        imperv_frac = 0
        imperv_area = 0
    
    # 加入 grid 属性表（wide 格式）
    grid[f"impervious_area_{year}"] = imperv_area

    # long 格式
    all_years.append(pd.DataFrame({
        "grid_id": grid["grid_id"],
        "year": year,
        "impervious_area_m2": imperv_area
    }))

# === Step 3. 导出 ===

# ① CSV / long format
df_out = pd.concat(all_years, ignore_index=True)
df_out.to_csv(out_csv, index=False)
print(f"💾 Saved yearly impervious CSV → {out_csv}")

# ② GeoPackage / wide format
grid.to_file(out_gpkg, driver="GPKG")
print(f"🗺️ Saved GeoPackage → {out_gpkg}")


✅ Loaded grid: 410620 cells


Processing CLCD yearly impervious area:   0%|          | 0/11 [00:00<?, ?it/s]

  → 2012: CLCD_v01_2012_albert_jiangsu_albers.tif


Processing CLCD yearly impervious area:   9%|▉         | 1/11 [06:40<1:06:47, 400.75s/it]

  → 2013: CLCD_v01_2013_albert_jiangsu_albers.tif


...

  → 2021: CLCD_v01_2021_albert_jiangsu_albers.tif


Processing CLCD yearly impervious area:  91%|█████████ | 10/11 [1:09:05<07:04, 424.56s/it]

  → 2022: CLCD_v01_2022_albert_jiangsu_albers.tif


Processing CLCD yearly impervious area: 100%|██████████| 11/11 [1:16:13<00:00, 415.75s/it]


💾 Saved yearly impervious CSV → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_impervious_2012_2022.csv
🗺️ Saved GeoPackage → /Users/wangze/Dropbox/Emi/LandControl/geodata/Jiangsu_CGCS2000/Jiangsu_grid_impervious_2012_2022.gpkg
